<a href="https://colab.research.google.com/github/Somrat390/NLP/blob/main/Bag_of_word.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q spacy

In [5]:
import spacy

nlp = spacy.load('en_core_web_sm')

def preprocess(text):
  doc = nlp(text)
  filter_content = []
  for token in doc:
    if token.is_stop or token.is_punct:
      continue
    filter_content.append(token.lemma_)
  return ' '.join(filter_content)

preprocess('I like to ate chicken')


'like ate chicken'

In [6]:
corpus = {
    'Thor ate pizza',
    'Loki is tall',
    'Loki is eating Pizza'
}

In [7]:
corpus_preprocess = [preprocess(text) for text in corpus]
corpus_preprocess

['Loki eat Pizza', 'thor eat pizza', 'Loki tall']

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

v = CountVectorizer(ngram_range=(1,3))
v.fit(corpus_preprocess)
v.vocabulary_

{'loki': 2,
 'eat': 0,
 'pizza': 6,
 'loki eat': 3,
 'eat pizza': 1,
 'loki eat pizza': 4,
 'thor': 8,
 'thor eat': 9,
 'thor eat pizza': 10,
 'tall': 7,
 'loki tall': 5}

In [10]:
v.transform(['Thor eat pizza']).toarray()

array([[1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1]])

In [11]:
import pandas as pd

df = pd.read_json('news_dataset.json')

df.head()

,text,category
0,Watching Schrödinger's Cat Die University of C...,SCIENCE
1,WATCH: Freaky Vortex Opens Up In Flooded Lake,SCIENCE
2,Entrepreneurs Today Don't Need a Big Budget to...,BUSINESS
3,These Roads Could Recharge Your Electric Car A...,BUSINESS
4,Civilian 'Guard' Fires Gun While 'Protecting' ...,CRIME


In [12]:
df.shape

(12695, 2)

In [13]:
df.category.value_counts()

,count
category,
BUSINESS,4254
SPORTS,4167
CRIME,2893
SCIENCE,1381


In [15]:
min_sample = 1381

df_business = df[df.category=='BUSINESS'].sample(min_sample, random_state=2022)
df_sports = df[df.category=='SPORTS'].sample(min_sample, random_state=2022)
df_crime = df[df.category=='CRIME'].sample(min_sample, random_state=2022)
df_science = df[df.category=='SCIENCE'].sample(min_sample, random_state=2022)


In [16]:
df_balance = pd.concat([df_business,df_sports,df_crime,df_science], axis=0)
df_balance.category.value_counts()

,count
category,
BUSINESS,1381
SPORTS,1381
CRIME,1381
SCIENCE,1381


In [17]:
df_balance['category_num'] = df_balance.category.map(
    {
        'BUSINESS': 0,
        'SPORTS': 1,
        'CRIME': 2,
        'SCIENCE': 3
    }
)

In [18]:
df_balance.head()

,text,category,category_num
11967,GCC Business Leaders Remain Confident in the F...,BUSINESS,0
2912,From the Other Side; an Honest Review from Emp...,BUSINESS,0
3408,"Mike McDerment, CEO of FreshBooks, Talks About...",BUSINESS,0
502,How to Market Your Business While Traveling th...,BUSINESS,0
5279,How to Leverage Intuition in Decision-making I...,BUSINESS,0


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_balance.text,
    df_balance.category_num,
    test_size = 0.2,
    stratify = df_balance.category_num,
    random_state = 2022
)

In [20]:
print(X_train.shape)
X_train.head()

(4419,)


,text
7589,Ovulating Women Prefer Images of Penetration O...
10442,Scientists Discover Spooky Influence On Baby N...
8792,Olympic Race Walker Steps Up To Propose To His...
1733,Beloved Bipedal Bear Named Pedals Believed Kil...
2526,"Elizabeth Smart Gave Birth To Baby Girl, Fathe..."


In [21]:
y_test.shape

(1105,)

In [22]:
y_test.value_counts()

,count
category_num,
1,277
0,276
3,276
2,276


In [24]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

clf = Pipeline(
    [
        ('vectorizer_bow', CountVectorizer()),
        ('Multi NB', MultinomialNB())
    ]
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.75      0.87      0.81       276
           1       0.93      0.80      0.86       277
           2       0.83      0.90      0.86       276
           3       0.90      0.80      0.85       276

    accuracy                           0.84      1105
   macro avg       0.85      0.84      0.84      1105
weighted avg       0.85      0.84      0.84      1105



In [26]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

clf = Pipeline(
    [
        ('vectorizer_bow', CountVectorizer(ngram_range=(1,2))),
        ('Multi NB', MultinomialNB())
    ]
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.69      0.90      0.78       276
           1       0.95      0.74      0.83       277
           2       0.82      0.88      0.85       276
           3       0.92      0.78      0.84       276

    accuracy                           0.82      1105
   macro avg       0.85      0.82      0.83      1105
weighted avg       0.85      0.82      0.83      1105



In [27]:
df_balance['preprocessed_text'] = df_balance.text.apply(preprocess)

In [30]:
df_balance.head()

,text,category,category_num,preprocessed_text
11967,GCC Business Leaders Remain Confident in the F...,BUSINESS,0,GCC Business leader remain confident Face Regi...
2912,From the Other Side; an Honest Review from Emp...,BUSINESS,0,Honest Review Employees wake morning love impo...
3408,"Mike McDerment, CEO of FreshBooks, Talks About...",BUSINESS,0,Mike McDerment CEO FreshBooks Talks give build...
502,How to Market Your Business While Traveling th...,BUSINESS,0,market business travel World recently amazing ...
5279,How to Leverage Intuition in Decision-making I...,BUSINESS,0,leverage intuition decision making feel safe r...


In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    df_balance.preprocessed_text,
    df_balance.category_num,
    test_size = 0.2,
    stratify = df_balance.category_num,
    random_state = 2022
)

In [31]:
clf = Pipeline(
    [
        ('vectorizer_bow', CountVectorizer(ngram_range=(1,2))),
        ('Multi NB', MultinomialNB())
    ]
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.78      0.88      0.83       276
           1       0.94      0.81      0.87       277
           2       0.82      0.91      0.86       276
           3       0.91      0.82      0.86       276

    accuracy                           0.85      1105
   macro avg       0.86      0.85      0.85      1105
weighted avg       0.86      0.85      0.85      1105

